In [ ]:
import json
import re
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path


# from lecture 6
# create requirements.txt by using pip freeze in your repo

### Task 1

In [ ]:
USER = "Bruker"
BASE_DIR = Path("C:\\", "Users", USER, "Documents", "folder_for_saving", "d2_data")

def read_file(filepath: str) -> tuple(str, str):
    """
    Opens and reads file. Checking encoding type between UTF-8 and ISO-8859-1

    ARGS: file path
    
    RETURNS: tuple containing strings for file text content and encoding.
    """

    # create a path object
    path = Path(filepath)

    # tries opening with UTF-8 encoding first and with latin-1/ISO-8859-1 second
    # This should correctly decode files encoded with UTF and ISO-8859-1
    # does not always return correct encoding as some non conflicting ISO-8859-1 files
    # can be opened in UTF-8
    try:
        filedata = open(path, encoding="UTF-8").read(), "UTF-8"
        return filedata
    except:
        filedata = open(path, encoding="ISO-8859-1").read(), "ISO-8859-1"
        return filedata


# test (not sure if the generated files from generate_data_for_d2 is the same so check generation log if encoding is correct)
# mine was. Giving data and UTF-8 encoding which is correct
# print(read_file(r"C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026\science_building\POS_02\register_sales.log")[1])

# for me this should be ISO-8859-1. It is not. 
# print(read_file(r"C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026\science_building\POS_03\register_sales.log"))
# print(read_file(r"{BASE_DIR}\2026\science_building\POS_03\register_sales.log"))

def parse_log(filecontent: str) -> list[dict]:
    """
    Reads files line by line and checks if lines are a valid log format and then extracts
    info from that log entry.

    ARGS: Content of a register_sales.log file.

    RETURNS: A tuple list containing dictionaries where each dictionary is a valid transaction

    WARNING: Does not handle tuples from read_file
    """
    valid_sales = []
    # Set up regex pattern for the that should be matched from log
    pattern = re.compile(
        r"\[(?P<timestamp>[\d\-]+\s[\w:]+)\] "
        r"\[(?P<register_id>POS_\d+)\] "
        r"item=(?P<item>\S+) "
        r"price=(?P<price>\S+) "
        r"(?P<currency>\w+) "
        r"\((?P<status>\w+)\)$",
        re.MULTILINE
    )

    # Creates a list of dictionaries where each dictionary will be a valid transaction by
    # Iterating over the lines in filecontent, as well as converting price to a float
    valid_sales = [{**match.groupdict(), "price": float(match.group("price"))} for match in pattern.finditer(filecontent)]
    return(valid_sales)

# Test
# parse_log(read_file(r"C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026\science_building\POS_03\register_sales.log")[0])
print(parse_log(read_file(Path(BASE_DIR, "2026", "science_building", "POS_03", "register_sales.log"))[0]))



[{'timestamp': '2026-10-15 07:02:40', 'register_id': 'POS_03', 'item': 'Sandwich', 'price': 7.58, 'currency': 'EUR', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:04:09', 'register_id': 'POS_03', 'item': 'Muffin', 'price': 3.17, 'currency': 'NOK', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:04:44', 'register_id': 'POS_03', 'item': 'Sandwich', 'price': 7.01, 'currency': 'EUR', 'status': 'REFUND'}, {'timestamp': '2026-10-15 07:06:25', 'register_id': 'POS_03', 'item': 'Latte', 'price': 4.54, 'currency': 'USD', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:08:52', 'register_id': 'POS_03', 'item': 'Croissant', 'price': 3.49, 'currency': 'EUR', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:09:20', 'register_id': 'POS_03', 'item': 'Sandwich', 'price': 7.64, 'currency': 'NOK', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:11:24', 'register_id': 'POS_03', 'item': 'Cappuccino', 'price': 4.16, 'currency': 'EUR', 'status': 'CANCELLED'}, {'timestamp': '2026-10-15 07:13:18

### Task 2

In [ ]:
def find_log_files(path: str) -> list[Path]:
    """Finds all .log files in target directory and sub directories """
    # Convert to Path object
    folder = Path(path)

    logs = []

    # iterate over items in directory
    for child in folder.iterdir():
        if child.is_file() and str(child)[-4:] == ".log":
            logs.append(child)  # if child is a .log file add Path to logs

        elif child.is_dir():
            # if child is a directory call function and add elements to logs via extend
            logs.extend(find_log_files(child))

    return logs

# Test
# print(find_log_files(r"C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026"))

# 
paths = find_log_files(BASE_DIR)

for path in paths:
    parse_log(read_file(path)[0])
    print("parsed log for", path)

def get_active_registers(data: str) -> set[str]:
    """
    Gets a set of active registers from a nested json dictionary
    ARGS: json dictionary
    RETURNS: set of active registers    
    """
    active_registers = set()
    if isinstance(data, dict):
        if data.get("active") is True and isinstance(data.get("id"), str):
            active_registers.add(data["id"])
        for value in data.values():
            active_registers.update(get_active_registers(value))

    elif isinstance(data, list):
        for item in data:
            active_registers.update(get_active_registers(item))

    return active_registers

with  open(Path(BASE_DIR, "store_config.json"), "r") as json_file:
    config = json.load(json_file)

print(get_active_registers(config))


[{'timestamp': '2026-10-15 07:02:20', 'register_id': 'POS_01', 'item': 'Espresso', 'price': 2.66, 'currency': 'USD', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:04:26', 'register_id': 'POS_01', 'item': 'Croissant', 'price': 4.21, 'currency': 'EUR', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:07:06', 'register_id': 'POS_01', 'item': 'Latte', 'price': 4.88, 'currency': 'EUR', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:08:46', 'register_id': 'POS_01', 'item': 'Cappuccino', 'price': 4.03, 'currency': 'EUR', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:10:47', 'register_id': 'POS_01', 'item': 'Croissant', 'price': 3.7, 'currency': 'USD', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:11:49', 'register_id': 'POS_01', 'item': 'Espresso', 'price': 2.98, 'currency': 'NOK', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:12:19', 'register_id': 'POS_01', 'item': 'Espresso', 'price': 2.51, 'currency': 'EUR', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:14:2

### Task 3

In [ ]:
def removing_price_outliers(input_price_list: list) -> np.ndarray:
    """
    Converts a list of price values to a 1D NumPy array and replaces all hardware glitch outliers
    with the median of the valid prices.

    Args:
        input_price_list: A list of price values.
    Returns:
        cleaned_price_array: A 1D NumPy array with valid prices.
    """

    # Convert the input price data list to a 1D NumPy float array
    input_price_array = np.array(input_price_list)

    # Calculate the mean and standard deviation of the array with prices
    mean = np.mean(input_price_array)
    sigma = np.std(input_price_array)

    # Find all valid prices to later calculate the median of the valid prices
    valid_prices = []
    for price in input_price_array:
        if abs(price - mean) <= 3 * sigma:
            valid_prices.append(price)

    # Calculate the median of the valid prices
    median = np.median(valid_prices)

    # Replace all hardware glitch outliers with the median of the valid prices
    cleaned_price_array = np.where(abs(input_price_array - mean) > 3 * sigma, median, input_price_array)

    return cleaned_price_array


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Calculates the refund ratio"""

    # We have 10**(-6) in the denominator to prevent zero-devision errors
    refund_ratio = refund_count / (sales_count + 10**(-6))

    return refund_ratio

### Task 4

### Task 5

In [ ]:
# starting on putting it all together
paths = find_log_files(BASE_DIR)

with  open(Path(BASE_DIR, "store_config.json"), "r") as json_file:
    config = json.load(json_file)

print(get_active_registers(config))
for path in paths:
    parse_log(read_file(path)[0])
    print("parsed log for", path)

parsed log for C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026\library_cafe\POS_01\register_sales.log
UTF-8
parsed log for C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026\library_cafe\POS_02\register_sales.log
UTF-8
parsed log for C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026\library_cafe\POS_03\register_sales.log
UTF-8
parsed log for C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026\science_building\POS_01\register_sales.log
UTF-8
parsed log for C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026\science_building\POS_02\register_sales.log
UTF-8
parsed log for C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026\science_building\POS_03\register_sales.log
UTF-8
parsed log for C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026\student_union\POS_01\register_sales.log
UTF-8
parsed log for C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026\student_union\POS_02\register_sales.log
UTF-8
parsed log for C:\Users\Bruker\Documents\folder_fo